# PTB-XL 12-lead ECG — Multi-paper research pipeline

Ek notebook, chaar papers ke recipes combine karke. Har stage kis paper se aaya hai:

| Stage | Base paper | Is notebook mein kya liya |
|---|---|---|
| 1. Data + labels + split | Wagner et al. 2020 (PTB-XL, *Sci. Data*) | `strat_fold` split (1–8 train, 9 val, 10 test), SCP label aggregation |
| 2. Preprocessing | Strodthoff et al. 2021 (PTB-XL benchmark, *IEEE JBHI*) | 100 Hz, StandardScaler fit only on train, 2.5 s random windows |
| 3. Augmentation | Mehari & Strodthoff 2022 (*Comput. Biol. Med.*) | Random Resized Crop + Time Out (best pair in their ablation), physio-noise |
| 4a. Supervised models | Strodthoff 2021 | `xresnet1d101`, `inception1d`, concat-pool head, AdamW + 1-cycle, BCE |
| 4b. SSL #1 — CPC | Mehari & Strodthoff 2022 | 4FC encoder + 2-layer LSTM, predict 12 steps, 128 negatives, two-step finetune + discriminative LRs |
| 4c. SSL #2 — Masked Transformer | Zhou et al. (MTECG) | segment tokens, 25% masking, per-segment-norm target, 1-layer decoder, layer-wise LR decay 0.6, DropPath 0.4 |
| 5. Evaluation | Strodthoff 2021 | sliding-window TTA, term-centric macro AUC, F<sub>β=2</sub>/G<sub>β=2</sub>, bootstrap 95% CI, paired significance, ensemble |

**Reference numbers (task `all`, 71 labels, macro AUC on fold 10):**
- supervised xresnet1d101 ≈ **0.925** (Strodthoff 2021)
- CPC pretrained on PTB-XL only → finetuned ≈ **0.9395**; pretrained on CinC2020+PTB-XL ≈ **0.9418** (Mehari 2022)
- MTECG reports macro-F1, not AUC, and pretrained on an *external* corpus for PTB-XL — PTB-XL-only pretraining yahan weaker ho sakta hai. Extra unlabeled ECG (e.g. Chapman) ke liye `EXTRA_PRETRAIN_NPY` hook hai.

**Run order:** Config cell set karo → *Run All*. Har experiment apne predictions `OUT_DIR` mein save karta hai; dobara run karne pe trained experiments skip ho jaate hain (`FORCE_RETRAIN=False`). Pretraining checkpoints resume hote hain.

Pehli baar `QUICK_DEBUG = True` karke chalao (minutes mein pura pipeline check), phir `False`.

In [1]:
# pip install wfdb torch scikit-learn pandas scipy tqdm
import os, sys, ast, json, math, time, random, copy, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import wfdb
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, sosfiltfilt
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=UserWarning)
try:
    display
except NameError:
    display = print
print('torch', torch.__version__, '| wfdb', wfdb.__version__)

torch 2.15.0+cu134 | wfdb 5.1.0


c:\Users\DELL-DAEMON\Documents\Github\XAI_Cardio_Research\coding\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Config
Sirf yeh cell edit karna hai.

In [2]:
DATA_PATH = '../PTBXL dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'

TASK = 'all'            # 'all' (71) | 'diagnostic' (44) | 'subdiagnostic' (23) | 'superdiagnostic' (5) | 'form' (19) | 'rhythm' (12)
SAMPLING_RATE = 100     # all four papers benchmark PTB-XL at 100 Hz
SEED = 42
QUICK_DEBUG = True     # True -> 1-2 epochs everywhere, pipeline check only
FORCE_RETRAIN = False   # True -> ignore saved predictions and retrain

RUN = {
    'xresnet1d101':     True,   # Strodthoff baseline (exact protocol)
    'xresnet1d101_aug': True,   # same + Mehari RRC/TimeOut augmentation (ablation)
    'inception1d':      True,   # Strodthoff baseline
    'cpc':              True,   # Mehari: CPC pretrain -> two-step finetune
    'mtecg':            True,   # MTECG: masked-transformer pretrain -> finetune
}

# Preprocessing
BANDPASS = None          # None = benchmark protocol (no filter). Ablation: (0.5, 40.0)
STANDARDIZE = 'global'   # 'global' = benchmark (one scaler over all leads) | 'per_lead'

# Optional extra *unlabeled* ECGs for SSL pretraining only (never labels, never PTB-XL test fold).
# Each entry: path to .npy of shape (N, 12, 10*SAMPLING_RATE), raw mV, same lead order as PTB-XL.
EXTRA_PRETRAIN_NPY = []

CFG_CNN = dict(epochs=50, bs=128, lr=1e-2, wd=1e-2, window_s=2.5)                 # Strodthoff repo defaults
CFG_CPC = dict(d=512, steps=12, n_neg=128, n_anchor=128,                       # n_anchor: timesteps per sequence used in the loss
               pretrain_epochs=200, pretrain_bs=64, pretrain_lr=1e-4, wd=1e-3,     # Mehari: 10 s input, constant LR
               head_epochs=50, full_epochs=20, ft_bs=128, ft_lr=1e-3, window_s=2.5)
CFG_MTECG = dict(patch_s=0.05, dim=192, depth=12, heads=3,                         # MTECG-T, 50 ms segments
                 dec_dim=128, dec_depth=1, dec_heads=4, mask_ratio=0.25, target='norm',
                 pretrain_epochs=800, pretrain_warmup=40, pretrain_bs=256, pretrain_lr=1e-3, pretrain_wd=0.05,
                 ft_epochs=50, ft_warmup=5, ft_bs=256, ft_lr=1e-3, ft_wd=0.05, drop_path=0.4, layer_decay=0.6)
# MTECG paper pretrains 1600 epochs and reports saturation near ~1000; 800 is a time/accuracy trade-off.

N_BOOTSTRAP = 1000       # Strodthoff used 10,000; 1000 is enough for 95% CIs to ~3rd decimal
EVAL_BS = 256

CACHE_DIR = Path('./ptbxl_cache')
OUT_DIR = Path('./ptbxl_runs') / f'{TASK}_{SAMPLING_RATE}hz'
CACHE_DIR.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)

if QUICK_DEBUG:
    CFG_CNN.update(epochs=2)
    CFG_CPC.update(pretrain_epochs=2, head_epochs=2, full_epochs=1)
    CFG_MTECG.update(pretrain_epochs=2, pretrain_warmup=1, ft_epochs=2, ft_warmup=1)
    N_BOOTSTRAP = 50
    OUT_DIR = OUT_DIR.parent / (OUT_DIR.name + '_debug'); OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

def autocast():
    return torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP) if USE_AMP else torch.autocast('cpu', enabled=False)

def make_scaler():
    enabled = USE_AMP and AMP_DTYPE == torch.float16
    try:    return torch.amp.GradScaler('cuda', enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)

seed_everything(SEED)
print('device:', DEVICE, '| AMP:', USE_AMP, AMP_DTYPE if USE_AMP else '')

device: cuda | AMP: True torch.bfloat16


## Stage 1 — Data, labels, split  *(Wagner 2020 + Strodthoff 2021)*

- Raw WFDB signals ek baar padh ke `.npy` cache ho jaate hain (pehli baar kuch minute lagenge).
- Label aggregation benchmark code jaisa hi: `scp_codes` ke keys, `scp_statements.csv` se diagnostic/form/rhythm filter, sub/super class mapping. Likelihood threshold **nahi** lagaya (benchmark bhi nahi lagata).
- Sirf woh records rakhe jaate hain jinke paas selected task ka ≥1 label hai.
- Split: folds 1–8 train, 9 val, 10 test. Folds patient-wise hain → leakage nahi.

In [4]:
def load_raw(path, sr):
    df = pd.read_csv(os.path.join(path, 'ptbxl_database.csv'), index_col='ecg_id')
    df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)
    cache = CACHE_DIR / f'raw_{sr}hz.npy'
    if cache.exists():
        X = np.load(cache, mmap_mode=None)
        if len(X) == len(df):
            return df, X
    files = df['filename_lr'] if sr == 100 else df['filename_hr']
    def _read(f): return wfdb.rdsamp(os.path.join(path, f))[0].astype(np.float32)
    with ThreadPoolExecutor(max_workers=8) as ex:
        sigs = list(tqdm(ex.map(_read, files), total=len(files), desc=f'reading WFDB @{sr}Hz'))
    X = np.stack(sigs)                     # (N, L, 12)
    np.save(cache, X)
    return df, X

def aggregate_labels(df, path, task):
    agg = pd.read_csv(os.path.join(path, 'scp_statements.csv'), index_col=0)
    if task in ('diagnostic', 'subdiagnostic', 'superdiagnostic'):
        dag = agg[agg.diagnostic == 1.0]
        col = {'diagnostic': None, 'subdiagnostic': 'diagnostic_subclass', 'superdiagnostic': 'diagnostic_class'}[task]
        def f(d):
            out = []
            for k in d:
                if k in dag.index:
                    v = k if col is None else dag.loc[k, col]
                    if isinstance(v, str) or not pd.isna(v): out.append(v)
            return sorted(set(out))
    elif task in ('form', 'rhythm'):
        sel = agg[agg[task] == 1.0]
        def f(d): return sorted({k for k in d if k in sel.index})
    elif task == 'all':
        def f(d): return sorted(set(d.keys()))
    else:
        raise ValueError(task)
    return df['scp_codes'].apply(f)

df_all, X_raw = load_raw(DATA_PATH, SAMPLING_RATE)
labels_all = aggregate_labels(df_all, DATA_PATH, TASK)
has_label = labels_all.apply(len).values > 0
folds_all = df_all['strat_fold'].values

mlb = MultiLabelBinarizer()
mlb.fit(labels_all[has_label])
CLASSES = list(mlb.classes_)
N_CLASSES = len(CLASSES)
Y_all = mlb.transform(labels_all).astype(np.float32)

idx_tr = np.where(has_label & (folds_all <= 8))[0]
idx_va = np.where(has_label & (folds_all == 9))[0]
idx_te = np.where(has_label & (folds_all == 10))[0]
print(f'records: {len(df_all)} | with {TASK} label: {has_label.sum()} | classes: {N_CLASSES}')
print(f'train {len(idx_tr)} | val {len(idx_va)} | test {len(idx_te)} | signal shape {X_raw.shape[1:]}')

records: 21837 | with all label: 21837 | classes: 71
train 17441 | val 2193 | test 2203 | signal shape (1000, 12)


## Stage 2 — Preprocessing  *(Strodthoff 2021)*

- Benchmark protocol: **koi filter nahi**, ek global `StandardScaler` jo sirf train-fold values pe fit hota hai. `BANDPASS` / `STANDARDIZE='per_lead'` ablation ke liye diye hain — paper mein inka claim nahi hai, isliye baseline ke liye default mat badlo.
- Scaler sab records pe apply hota hai (SSL pretraining unlabeled train-fold records bhi use karta hai).
- Layout `(N, 12, L)` — PyTorch Conv1d channels-first.

In [5]:
def bandpass(X, fs, lo, hi, order=3):
    sos = butter(order, [lo, hi], btype='band', fs=fs, output='sos')
    return sosfiltfilt(sos, X, axis=1).astype(np.float32)

X_proc = X_raw.copy()
if BANDPASS is not None:
    X_proc = bandpass(X_proc, SAMPLING_RATE, *BANDPASS)

fit_ref = X_proc[idx_tr]                                   # labelled train folds only
if STANDARDIZE == 'global':
    MU, SD = float(fit_ref.mean()), float(fit_ref.std())
elif STANDARDIZE == 'per_lead':
    MU, SD = fit_ref.mean(axis=(0, 1), keepdims=True), fit_ref.std(axis=(0, 1), keepdims=True)
else:
    raise ValueError(STANDARDIZE)
del fit_ref

def standardize(X):  # X: (N, L, 12) raw
    return ((X - MU) / (SD + 1e-8)).astype(np.float32)

X_std = np.ascontiguousarray(standardize(X_proc).transpose(0, 2, 1))   # (N, 12, L)
del X_proc
SIG_LEN = X_std.shape[-1]

# keep signals on GPU if they fit (~1 GB at 100 Hz) -> no DataLoader bottleneck
STORE = DEVICE if (DEVICE.type == 'cuda' and X_std.nbytes < 4e9) else torch.device('cpu')
X_t = torch.from_numpy(X_std).to(STORE)
Y_t = torch.from_numpy(Y_all).to(STORE)

def subset(idx): return X_t[torch.as_tensor(idx, device=STORE)], Y_t[torch.as_tensor(idx, device=STORE)]
Xtr, Ytr = subset(idx_tr); Xva, Yva = subset(idx_va); Xte, Yte = subset(idx_te)
Y_val_np, Y_test_np = Y_all[idx_va], Y_all[idx_te]

# Unlabeled pool for SSL: every train-fold record (also those without a label for this TASK) + optional extra data
idx_unl = np.where(folds_all <= 8)[0]
idx_unl_val = np.where(folds_all == 9)[0]
X_unl = X_t[torch.as_tensor(idx_unl, device=STORE)]
for p in EXTRA_PRETRAIN_NPY:
    ext = np.load(p).astype(np.float32)                    # (N, 12, L) raw
    assert ext.shape[1] == 12 and ext.shape[2] == SIG_LEN, f'{p}: expected (N,12,{SIG_LEN}), got {ext.shape}'
    if BANDPASS is not None: ext = bandpass(ext.transpose(0, 2, 1), SAMPLING_RATE, *BANDPASS).transpose(0, 2, 1)
    ext = standardize(ext.transpose(0, 2, 1)).transpose(0, 2, 1)
    X_unl = torch.cat([X_unl, torch.from_numpy(np.ascontiguousarray(ext)).to(STORE)])
X_unl_val = X_t[torch.as_tensor(idx_unl_val, device=STORE)]
print('SSL pool:', tuple(X_unl.shape), '| SSL val:', tuple(X_unl_val.shape), '| stored on', STORE)

SSL pool: (17441, 12, 1000) | SSL val: (2193, 12, 1000) | stored on cuda


## Stage 3 — Windowing + augmentation  *(Strodthoff 2021, Mehari & Strodthoff 2022)*

- **Sliding-window training**: har step pe har record se random 2.5 s crop → itself an augmentation.
- **Mehari transformations** (batched on GPU): Random Resized Crop (50–100%), Time Out (≤50% zeroed) — unke SimCLR ablation mein best pair. Physio noise (baseline wander, EMG, baseline shift) bhi diya hai. Powerline (50/60 Hz) 100 Hz sampling pe Nyquist pe hai, isliye skip.

In [6]:
def aug_rrc(x, lo=0.5, hi=1.0):
    B, C, L = x.shape
    scale = torch.empty(B, 1, device=x.device).uniform_(lo, hi)
    start = torch.rand(B, 1, device=x.device) * (1 - scale)
    u = torch.linspace(0, 1, L, device=x.device)[None]
    gx = (start + scale * u) * 2 - 1                                   # (B, L) in [-1, 1]
    grid = torch.stack([gx, torch.zeros_like(gx)], -1)[:, None]         # (B, 1, L, 2)
    return F.grid_sample(x[:, :, None], grid, mode='bilinear', align_corners=True)[:, :, 0]

def aug_timeout(x, lo=0.0, hi=0.5):
    B, C, L = x.shape
    t = torch.empty(B, 1, device=x.device).uniform_(lo, hi)
    s = torch.rand(B, 1, device=x.device) * (1 - t)
    pos = torch.linspace(0, 1, L, device=x.device)[None]
    keep = ~((pos >= s) & (pos < s + t))
    return x * keep[:, None].to(x.dtype)

def aug_gauss(x, std=0.05):          return x + std * torch.randn_like(x)
def aug_channel_resize(x, b=3.0):    return x * (b ** torch.empty(x.shape[0], x.shape[1], 1, device=x.device).uniform_(-1, 1))

def aug_baseline_wander(x, fs=SAMPLING_RATE, amp=0.1, fmax=0.5, n=3):
    B, C, L = x.shape
    t = torch.arange(L, device=x.device)[None, None] / fs
    out = x
    for _ in range(n):
        f = torch.rand(B, 1, 1, device=x.device) * fmax
        ph = torch.rand(B, C, 1, device=x.device) * 2 * math.pi
        a = torch.rand(B, C, 1, device=x.device) * amp
        out = out + a * torch.sin(2 * math.pi * f * t + ph)
    return out

def aug_emg(x, std=0.05):            return x + std * torch.rand(x.shape[0], x.shape[1], 1, device=x.device) * torch.randn_like(x)

def aug_baseline_shift(x, amp=0.2, max_frac=0.3):
    B, C, L = x.shape
    t = torch.rand(B, 1, device=x.device) * max_frac
    s = torch.rand(B, 1, device=x.device) * (1 - t)
    pos = torch.linspace(0, 1, L, device=x.device)[None]
    step = ((pos >= s) & (pos < s + t)).to(x.dtype)[:, None]
    return x + step * (torch.rand(B, C, 1, device=x.device) * 2 - 1) * amp

def _apply_p(fn, x, p):
    m = (torch.rand(x.shape[0], 1, 1, device=x.device) < p).to(x.dtype)
    return m * fn(x) + (1 - m) * x

def mehari_aug(x, p=0.5, physio=False):
    x = _apply_p(aug_rrc, x, p)
    x = _apply_p(aug_timeout, x, p)
    if physio:
        x = _apply_p(aug_baseline_wander, x, p * 0.5)
        x = _apply_p(aug_emg, x, p * 0.5)
        x = _apply_p(aug_baseline_shift, x, p * 0.5)
    return x

class WindowBatches:
    # Random fixed-length crops from full records, shuffled mini-batches, optional batched augmentation.
    def __init__(self, X, Y=None, bs=128, win=None, shuffle=True, aug=None, drop_last=True):
        self.X, self.Y, self.bs, self.shuffle, self.aug, self.drop_last = X, Y, bs, shuffle, aug, drop_last
        self.win = win or X.shape[-1]
    def __len__(self):
        n = len(self.X)
        return n // self.bs if self.drop_last else math.ceil(n / self.bs)
    def __iter__(self):
        n, L, C = len(self.X), self.X.shape[-1], self.X.shape[1]
        order = torch.randperm(n, device=self.X.device) if self.shuffle else torch.arange(n, device=self.X.device)
        for b in range(len(self)):
            idx = order[b * self.bs:(b + 1) * self.bs]
            xb = self.X[idx]
            if self.win < L:
                st = torch.randint(0, L - self.win + 1, (len(idx),), device=xb.device)
                ar = (st[:, None] + torch.arange(self.win, device=xb.device)[None])[:, None].expand(-1, C, -1)
                xb = torch.gather(xb, 2, ar)
            xb = xb.to(DEVICE, non_blocking=True)
            if self.aug is not None:
                xb = self.aug(xb)
            yield (xb, self.Y[idx].to(DEVICE, non_blocking=True)) if self.Y is not None else xb

_x = next(iter(WindowBatches(Xtr, Ytr, bs=4, win=int(2.5 * SAMPLING_RATE), aug=lambda z: mehari_aug(z, p=1.0, physio=True))))
print('window batch:', tuple(_x[0].shape), tuple(_x[1].shape))

window batch: (4, 12, 250) (4, 71)


## Stage 4a — Supervised architectures  *(Strodthoff 2021)*

- `xresnet1d101`: xresnet tricks (3-conv stem, AvgPool in shortcut, zero-init last BN), kernel size 5.
- `inception1d`: InceptionTime, depth 6, 32 filters, kernels 39/19/9, bottleneck 32, residual every 3 blocks.
- Dono ka head same: concat-pool (avg ⊕ max) → BN → Dropout 0.25 → Linear 128 → ReLU → BN → Dropout 0.5 → Linear.

In [7]:
class ConcatPoolHead(nn.Module):
    def __init__(self, nf, n_out, hidden=128, ps=(0.25, 0.5)):
        super().__init__()
        self.layers = nn.Sequential(
            nn.BatchNorm1d(2 * nf), nn.Dropout(ps[0]), nn.Linear(2 * nf, hidden), nn.ReLU(inplace=True),
            nn.BatchNorm1d(hidden), nn.Dropout(ps[1]), nn.Linear(hidden, n_out))
    def forward(self, x):
        return self.layers(torch.cat([F.adaptive_avg_pool1d(x, 1), F.adaptive_max_pool1d(x, 1)], 1).flatten(1))

class ConvBN(nn.Sequential):
    def __init__(self, ni, nf, ks=5, stride=1, act=True, zero_bn=False):
        bn = nn.BatchNorm1d(nf); nn.init.constant_(bn.weight, 0. if zero_bn else 1.)
        layers = [nn.Conv1d(ni, nf, ks, stride, padding=ks // 2, bias=False), bn]
        if act: layers.append(nn.ReLU(inplace=True))
        super().__init__(*layers)

class XResBlock(nn.Module):
    def __init__(self, expansion, ni, nf, stride=1, ks=5):
        super().__init__()
        nin, nout = ni * expansion, nf * expansion
        if expansion == 1:
            convs = [ConvBN(nin, nf, ks, stride), ConvBN(nf, nout, ks, act=False, zero_bn=True)]
        else:
            convs = [ConvBN(nin, nf, 1), ConvBN(nf, nf, ks, stride), ConvBN(nf, nout, 1, act=False, zero_bn=True)]
        self.convs = nn.Sequential(*convs)
        idp = []
        if stride != 1: idp.append(nn.AvgPool1d(2, ceil_mode=True))
        if nin != nout: idp.append(ConvBN(nin, nout, 1, act=False))
        self.idpath = nn.Sequential(*idp)
    def forward(self, x): return F.relu(self.convs(x) + self.idpath(x))

class XResNet1d(nn.Module):
    def __init__(self, expansion, layers, c_in=12, n_out=71, ks=5, ks_stem=5, stem=(32, 32, 64)):
        super().__init__()
        sz = [c_in, *stem]
        self.stem = nn.Sequential(*[ConvBN(sz[i], sz[i + 1], ks_stem, stride=2 if i == 0 else 1) for i in range(3)],
                                  nn.MaxPool1d(3, 2, 1))
        bs = [64 // expansion, 64, 128, 256, 512]
        blocks = []
        for i, l in enumerate(layers):
            for j in range(l):
                blocks.append(XResBlock(expansion, bs[i] if j == 0 else bs[i + 1], bs[i + 1],
                                        stride=1 if (i == 0 or j > 0) else 2, ks=ks))
        self.body = nn.Sequential(*blocks)
        self.head = ConcatPoolHead(bs[-1] * expansion, n_out)
    def forward(self, x): return self.head(self.body(self.stem(x)))

def xresnet1d50(**kw):  return XResNet1d(4, [3, 4, 6, 3], **kw)
def xresnet1d101(**kw): return XResNet1d(4, [3, 4, 23, 3], **kw)

class InceptionBlock1d(nn.Module):
    def __init__(self, ni, nb_filters=32, kss=(39, 19, 9), bottleneck=32):
        super().__init__()
        self.bottleneck = nn.Conv1d(ni, bottleneck, 1, bias=False) if bottleneck > 0 else nn.Identity()
        bi = bottleneck if bottleneck > 0 else ni
        self.convs = nn.ModuleList([nn.Conv1d(bi, nb_filters, k, padding=k // 2, bias=False) for k in kss])
        self.pool_conv = nn.Sequential(nn.MaxPool1d(3, 1, 1), nn.Conv1d(ni, nb_filters, 1, bias=False))
        self.bn = nn.BatchNorm1d((len(kss) + 1) * nb_filters)
    def forward(self, x):
        b = self.bottleneck(x)
        return F.relu(self.bn(torch.cat([c(b) for c in self.convs] + [self.pool_conv(x)], 1)))

class Shortcut1d(nn.Module):
    def __init__(self, ni, nf):
        super().__init__(); self.conv = nn.Conv1d(ni, nf, 1, bias=False); self.bn = nn.BatchNorm1d(nf)
    def forward(self, inp, out): return F.relu(out + self.bn(self.conv(inp)))

class Inception1d(nn.Module):
    def __init__(self, c_in=12, n_out=71, depth=6, nb_filters=32, kernel_size=40, bottleneck=32):
        super().__init__()
        kss = [kernel_size - 1, kernel_size // 2 - 1, kernel_size // 4 - 1]
        nf = (len(kss) + 1) * nb_filters
        self.blocks = nn.ModuleList([InceptionBlock1d(c_in if d == 0 else nf, nb_filters, kss, bottleneck) for d in range(depth)])
        self.shortcuts = nn.ModuleList([Shortcut1d(c_in if d == 0 else nf, nf) for d in range(depth // 3)])
        self.head = ConcatPoolHead(nf, n_out)
    def forward(self, x):
        res = x
        for d, blk in enumerate(self.blocks):
            x = blk(x)
            if d % 3 == 2:
                x = self.shortcuts[d // 3](res, x); res = x
        return self.head(x)

for name, m in [('xresnet1d101', xresnet1d101(n_out=N_CLASSES)), ('inception1d', Inception1d(n_out=N_CLASSES))]:
    print(f'{name}: {sum(p.numel() for p in m.parameters())/1e6:.2f}M params, out', tuple(m(torch.randn(2, 12, 250)).shape))

xresnet1d101: 33.56M params, out (2, 71)
inception1d: 0.52M params, out (2, 71)


## Stage 4b — Contrastive Predictive Coding  *(Mehari & Strodthoff 2022)*

- Encoder: per-timestep 4 × (Linear 512 + BN + ReLU) — 100 Hz pe strided conv ki zaroorat nahi.
- Context: 2-layer LSTM (512). Predictor: extra hidden layer + ReLU (unka "enhanced CPC").
- InfoNCE: 12 steps (0.12 s) aage predict, 128 negatives same sequence se.
- Finetune head: concat-pool (max ⊕ mean ⊕ last LSTM output) → 512 hidden, BN, dropout.

In [8]:
class CPCModel(nn.Module):
    def __init__(self, c_in=12, d=512, n_enc=4, lstm_layers=2, steps=12, n_out=None):
        super().__init__()
        layers, ni = [], c_in
        for _ in range(n_enc):
            layers += [nn.Conv1d(ni, d, 1, bias=False), nn.BatchNorm1d(d), nn.ReLU(inplace=True)]; ni = d
        self.encoder = nn.Sequential(*layers)
        self.rnn = nn.LSTM(d, d, lstm_layers, batch_first=True)
        self.d, self.steps = d, steps
        self.proj = nn.Sequential(nn.Linear(d, d), nn.ReLU(inplace=True), nn.Linear(d, d * steps))
        self.head = None
        if n_out is not None: self.add_head(n_out)

    def add_head(self, n_out, hidden=512, ps=(0.25, 0.5)):
        self.head = nn.Sequential(nn.BatchNorm1d(3 * self.d), nn.Dropout(ps[0]), nn.Linear(3 * self.d, hidden),
                                  nn.ReLU(inplace=True), nn.BatchNorm1d(hidden), nn.Dropout(ps[1]), nn.Linear(hidden, n_out))

    def _context(self, x):
        z = self.encoder(x).transpose(1, 2)                      # (B, T, d)
        with torch.autocast(device_type=x.device.type, enabled=False):
            c, _ = self.rnn(z.float())                           # LSTM in fp32 (AMP-safe)
        return z, c

    def cpc_loss(self, x, n_neg=128, n_anchor=128):
        # InfoNCE on a random subset of anchor timesteps per sequence (unbiased estimate of the full loss).
        # Predicting 12 x 512-d vectors for all 1000 timesteps was the memory bottleneck.
        z, c = self._context(x)
        B, T, D = z.shape
        with torch.autocast(device_type=x.device.type, enabled=False):
            z, c = z.float(), c.float()
            h = self.proj[1](self.proj[0](c))                                  # shared hidden (B, T, D)
            W, b = self.proj[2].weight, self.proj[2].bias                       # step k uses rows (k-1)*D : k*D
            loss, acc = 0., 0.
            for k in range(1, self.steps + 1):
                nA = min(n_anchor, T - k)
                t = torch.randint(0, T - k, (B, nA), device=x.device)
                hk = torch.gather(h, 1, t[:, :, None].expand(-1, -1, D))
                p = F.linear(hk, W[(k - 1) * D:k * D], b[(k - 1) * D:k * D])     # (B, nA, D)
                zp = torch.gather(z, 1, (t + k)[:, :, None].expand(-1, -1, D))
                pos = (p * zp).sum(-1, keepdim=True)                              # (B, nA, 1)
                neg_idx = torch.randint(0, T, (B, n_neg), device=x.device)        # negatives from the same sequence
                zn = torch.gather(z, 1, neg_idx[:, :, None].expand(-1, -1, D))    # (B, n_neg, D)
                logits = torch.cat([pos, torch.bmm(p, zn.transpose(1, 2))], 2).reshape(-1, 1 + n_neg)
                tgt = torch.zeros(len(logits), dtype=torch.long, device=x.device)
                loss = loss + F.cross_entropy(logits, tgt)
                acc = acc + (logits.argmax(1) == 0).float().mean()
        return loss / self.steps, acc / self.steps

    def forward(self, x):
        _, c = self._context(x)
        pooled = torch.cat([c.max(1)[0], c.mean(1), c[:, -1]], 1)
        return self.head(pooled)

_m = CPCModel(n_out=N_CLASSES)
print(f'CPC: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params | loss on random input:', round(_m.cpc_loss(torch.randn(2, 12, 200), 16)[0].item(), 3))

CPC: 9.24M params | loss on random input: 2.916


## Stage 4c — Masked Transformer (MTECG)  *(Zhou et al.)*

- 12 leads × 50 ms segment = ek token (paper: 25 samples @ 500 Hz = 5 samples @ 100 Hz → 10 s pe 200 tokens).
- Learnable positional embedding + aux (CLS) token. MTECG-T: D=192, 3 heads, 12 layers.
- Pretrain: 25% segments masked, 1-layer decoder (128-d, 4 heads), target = per-segment normalized signal.
- Finetune: global mean-pool of tokens → LayerNorm → Linear, DropPath 0.4, layer-wise LR decay 0.6.

In [9]:
class DropPath(nn.Module):
    def __init__(self, p=0.): super().__init__(); self.p = p
    def forward(self, x):
        if self.p == 0. or not self.training: return x
        keep = 1 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep

class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__(); self.h = heads; self.qkv = nn.Linear(dim, dim * 3); self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4., drop_path=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim); self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Linear(int(dim * mlp_ratio), dim))
        self.dp = DropPath(drop_path)
    def forward(self, x):
        x = x + self.dp(self.attn(self.norm1(x)))
        return x + self.dp(self.mlp(self.norm2(x)))

class MTECG(nn.Module):
    def __init__(self, c_in=12, seq_len=1000, patch=5, dim=192, depth=12, heads=3, drop_path=0.,
                 n_out=None, with_decoder=True, dec_dim=128, dec_depth=1, dec_heads=4):
        super().__init__()
        self.c_in, self.patch, self.T = c_in, patch, seq_len // patch
        self.patch_embed = nn.Linear(c_in * patch, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.T + 1, dim))
        dpr = torch.linspace(0, drop_path, depth).tolist()
        self.blocks = nn.ModuleList([Block(dim, heads, drop_path=dpr[i]) for i in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.with_decoder = with_decoder
        if with_decoder:
            self.dec_embed = nn.Linear(dim, dec_dim)
            self.mask_token = nn.Parameter(torch.zeros(1, 1, dec_dim))
            self.dec_pos = nn.Parameter(torch.zeros(1, self.T + 1, dec_dim))
            self.dec_blocks = nn.ModuleList([Block(dec_dim, dec_heads) for _ in range(dec_depth)])
            self.dec_norm = nn.LayerNorm(dec_dim)
            self.dec_pred = nn.Linear(dec_dim, c_in * patch)
        self.fc_norm = nn.LayerNorm(dim) if n_out else None
        self.head = nn.Linear(dim, n_out) if n_out else None
        self._init()

    def _init(self):
        for p in [self.cls_token, self.pos_embed] + ([self.mask_token, self.dec_pos] if self.with_decoder else []):
            nn.init.trunc_normal_(p, std=.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        if self.head is not None:
            nn.init.trunc_normal_(self.head.weight, std=2e-5)

    def patchify(self, x):                     # (B, C, L) -> (B, T, C*patch)
        B, C, L = x.shape
        return x[:, :, :self.T * self.patch].reshape(B, C, self.T, self.patch).permute(0, 2, 1, 3).reshape(B, self.T, C * self.patch)

    def encode(self, x, mask_ratio=0.):
        t = self.patch_embed(self.patchify(x)) + self.pos_embed[:, 1:]
        B, T, D = t.shape
        mask = ids_restore = None
        if mask_ratio > 0:
            n_keep = max(1, int(T * (1 - mask_ratio)))
            ids_shuffle = torch.argsort(torch.rand(B, T, device=x.device), 1)
            ids_restore = torch.argsort(ids_shuffle, 1)
            t = torch.gather(t, 1, ids_shuffle[:, :n_keep, None].expand(-1, -1, D))
            mask = torch.ones(B, T, device=x.device); mask[:, :n_keep] = 0
            mask = torch.gather(mask, 1, ids_restore)
        cls = (self.cls_token + self.pos_embed[:, :1]).expand(B, -1, -1)
        h = torch.cat([cls, t], 1)
        for blk in self.blocks: h = blk(h)
        return self.norm(h), mask, ids_restore

    def decode(self, h, ids_restore):
        h = self.dec_embed(h)
        B, _, D = h.shape
        mt = self.mask_token.expand(B, self.T + 1 - h.shape[1], -1)
        h_ = torch.cat([h[:, 1:], mt], 1)
        h_ = torch.gather(h_, 1, ids_restore[:, :, None].expand(-1, -1, D))
        h = torch.cat([h[:, :1], h_], 1) + self.dec_pos
        for blk in self.dec_blocks: h = blk(h)
        return self.dec_pred(self.dec_norm(h))[:, 1:]

    def mae_loss(self, x, mask_ratio=0.25, target='norm'):
        h, mask, ids = self.encode(x, mask_ratio)
        pred = self.decode(h, ids).float()
        tgt = self.patchify(x).float()
        if target == 'norm':
            mu, var = tgt.mean(-1, keepdim=True), tgt.var(-1, keepdim=True)
            tgt = (tgt - mu) / (var + 1e-6).sqrt()
        elif target == 'sqrt':
            tgt = tgt.sign() * tgt.abs().sqrt()
        loss = ((pred - tgt) ** 2).mean(-1)
        return (loss * mask).sum() / mask.sum()

    def forward(self, x):
        h, _, _ = self.encode(x, 0.)
        return self.head(self.fc_norm(h[:, 1:].mean(1)))

MT_PATCH = max(1, int(round(CFG_MTECG['patch_s'] * SAMPLING_RATE)))
_m = MTECG(seq_len=SIG_LEN, patch=MT_PATCH, n_out=N_CLASSES)
print(f'MTECG-T: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params | tokens: {SIG_LEN // MT_PATCH} | '
      f'mae loss on random input: {_m.mae_loss(torch.randn(2, 12, SIG_LEN)).item():.3f}')
del _m

MTECG-T: 5.66M params | tokens: 200 | mae loss on random input: 2.561


## Stage 5 — Training engine + evaluation  *(Strodthoff 2021)*

- **TTA**: record ko window-size segments mein kaato (stride = window/2), predictions aggregate karo — benchmark `max` use karta hai, Mehari CPC `mean`.
- **Model selection**: best validation macro AUC (fold 9). Test fold 10 sirf end mein.
- **Metrics**: term-centric macro AUC (primary), macro-F1 + F<sub>β=2</sub> / G<sub>β=2</sub> (PhysioNet/CinC challenge style) — thresholds *validation* pe tune, test pe apply.
- **Bootstrap**: same resampled indices sab models ke liye (paired). Woh resamples discard jinmein koi class ka positive na ho (benchmark rule).

In [10]:
@torch.no_grad()
def predict(model, X, window, stride=None, agg='max', bs=EVAL_BS):
    model.eval()
    L = X.shape[-1]; window = min(window, L); stride = stride or max(1, window // 2)
    starts = list(range(0, L - window + 1, stride))
    if starts[-1] != L - window: starts.append(L - window)
    out = []
    per = max(1, bs // len(starts))
    for i in range(0, len(X), per):
        xb = X[i:i + per].to(DEVICE, non_blocking=True)
        w = torch.stack([xb[:, :, s:s + window] for s in starts], 1).flatten(0, 1)
        with autocast():
            p = torch.sigmoid(model(w).float()).view(len(xb), len(starts), -1)
        out.append((p.max(1)[0] if agg == 'max' else p.mean(1)).cpu())
    return torch.cat(out).numpy()

def macro_auc(y, p):
    pos = y.sum(0); cols = (pos > 0) & (pos < len(y))
    return roc_auc_score(y[:, cols], p[:, cols], average='macro')

def per_class_auc(y, p):
    return np.array([roc_auc_score(y[:, j], p[:, j]) if 0 < y[:, j].sum() < len(y) else np.nan for j in range(y.shape[1])])

def tune_thresholds(y, p, grid=np.linspace(0.01, 0.99, 99)):
    th = np.zeros(y.shape[1])
    for j in range(y.shape[1]):
        yp = p[:, j][:, None] >= grid[None]
        tp = (yp & (y[:, j][:, None] == 1)).sum(0); fp = (yp & (y[:, j][:, None] == 0)).sum(0)
        fn = ((~yp) & (y[:, j][:, None] == 1)).sum(0)
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        th[j] = grid[f1.argmax()]
    return th

def threshold_metrics(y, yhat, beta=2.):
    y, yhat = y.astype(bool), yhat.astype(bool)
    tp, fp, fn = (y & yhat).sum(0), (~y & yhat).sum(0), (y & ~yhat).sum(0)
    f1 = np.mean(2 * tp / np.maximum(2 * tp + fp + fn, 1))
    w = 1. / np.maximum(y.sum(1, keepdims=True), 1)                        # challenge sample weighting
    TP, FP, FN = (w * (y & yhat)).sum(0), (w * (~y & yhat)).sum(0), (w * (y & ~yhat)).sum(0)
    fb = np.mean((1 + beta**2) * TP / np.maximum((1 + beta**2) * TP + FP + beta**2 * FN, 1e-12))
    gb = np.mean(TP / np.maximum(TP + FP + beta * FN, 1e-12))
    return dict(macro_F1=f1, F_beta2=fb, G_beta2=gb)

def bootstrap_indices(y, n, seed=SEED, max_tries_factor=50):
    rng = np.random.default_rng(seed); out, tries = [], 0
    while len(out) < n and tries < n * max_tries_factor:
        idx = rng.integers(0, len(y), len(y)); tries += 1
        if y[idx].sum(0).min() > 0: out.append(idx)
    if len(out) < n:
        print(f'  note: only {len(out)}/{n} resamples contain every class (rare labels); rest use classes present')
        out += [rng.integers(0, len(y), len(y)) for _ in range(n - len(out))]
    return out

def cosine_warmup(total, warmup):
    def f(step):
        if step < warmup: return (step + 1) / max(1, warmup)
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    return f

def train_classifier(model, *, name, param_groups, epochs, bs, win, sched='onecycle', warmup_epochs=0,
                     aug=None, eval_win=None, agg='max', betas=(0.9, 0.99), select_on='auc'):
    model.to(DEVICE)
    opt = torch.optim.AdamW(param_groups, betas=betas)
    it = WindowBatches(Xtr, Ytr, bs=bs, win=win, aug=aug)
    total = epochs * len(it)
    if sched == 'onecycle':
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in param_groups], total_steps=total,
                                                  pct_start=0.25, div_factor=25., final_div_factor=1e4)
    elif sched == 'cosine':
        sch = torch.optim.lr_scheduler.LambdaLR(opt, cosine_warmup(total, warmup_epochs * len(it)))
    else:
        sch = None
    scaler = make_scaler()
    best, best_state, hist = -1, None, []
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.
        for xb, yb in it:
            with autocast():
                loss = F.binary_cross_entropy_with_logits(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sch is not None: sch.step()
            run += loss.item()
        pv = predict(model, Xva, eval_win or win, agg=agg)
        auc = macro_auc(Y_val_np, pv)
        hist.append(dict(epoch=ep + 1, train_loss=run / len(it), val_auc=auc))
        flag = ''
        if auc > best:
            best, best_state, flag = auc, copy.deepcopy(model.state_dict()), ' *'
        print(f'[{name}] ep {ep+1:3d}/{epochs} loss {run/len(it):.4f} val macroAUC {auc:.4f}{flag} ({time.time()-t0:.0f}s)')
    model.load_state_dict(best_state)
    pd.DataFrame(hist).to_csv(OUT_DIR / f'{name}_history.csv', index=False)
    return model, best

RESULTS = {}   # name -> dict(val=probs, test=probs)
def run_experiment(name, fn):
    pv_f, pt_f = OUT_DIR / f'{name}_val.npy', OUT_DIR / f'{name}_test.npy'
    if pv_f.exists() and pt_f.exists() and not FORCE_RETRAIN:
        print(f'[{name}] loading saved predictions from {OUT_DIR}')
        RESULTS[name] = dict(val=np.load(pv_f), test=np.load(pt_f))
    else:
        seed_everything(SEED); t0 = time.time()
        pv, pt = fn()
        np.save(pv_f, pv); np.save(pt_f, pt)
        RESULTS[name] = dict(val=pv, test=pt)
        print(f'[{name}] done in {(time.time()-t0)/60:.1f} min')
    r = RESULTS[name]
    print(f'[{name}] val macroAUC {macro_auc(Y_val_np, r["val"]):.4f} | test macroAUC {macro_auc(Y_test_np, r["test"]):.4f}')

## Experiment 1 — Supervised baselines  *(Strodthoff 2021 protocol)*
AdamW, one-cycle LR 1e-2, wd 1e-2, batch 128, 50 epochs, 2.5 s windows, TTA max. `_aug` variant pe Mehari RRC + TimeOut (p=0.5) lagta hai.

In [11]:
WIN = int(CFG_CNN['window_s'] * SAMPLING_RATE)

def exp_cnn(arch, aug=None, name=None):
    def fn():
        model = {'xresnet1d101': xresnet1d101, 'inception1d': Inception1d}[arch](n_out=N_CLASSES)
        groups = [dict(params=model.parameters(), lr=CFG_CNN['lr'], weight_decay=CFG_CNN['wd'])]
        model, _ = train_classifier(model, name=name, param_groups=groups, epochs=CFG_CNN['epochs'], bs=CFG_CNN['bs'],
                                    win=WIN, sched='onecycle', aug=aug, agg='max', betas=(0.95, 0.99))
        torch.save(model.state_dict(), OUT_DIR / f'{name}.pt')
        return predict(model, Xva, WIN, agg='max'), predict(model, Xte, WIN, agg='max')
    return fn

if RUN['xresnet1d101']:
    run_experiment('xresnet1d101', exp_cnn('xresnet1d101', name='xresnet1d101'))
if RUN['xresnet1d101_aug']:
    run_experiment('xresnet1d101_aug', exp_cnn('xresnet1d101', aug=lambda x: mehari_aug(x, p=0.5), name='xresnet1d101_aug'))
if RUN['inception1d']:
    run_experiment('inception1d', exp_cnn('inception1d', name='inception1d'))

[xresnet1d101] loading saved predictions from ptbxl_runs\all_100hz_debug
[xresnet1d101] val macroAUC 0.8086 | test macroAUC 0.7935
[xresnet1d101_aug] loading saved predictions from ptbxl_runs\all_100hz_debug
[xresnet1d101_aug] val macroAUC 0.7986 | test macroAUC 0.7868
[inception1d] loading saved predictions from ptbxl_runs\all_100hz_debug
[inception1d] val macroAUC 0.8140 | test macroAUC 0.8096


## Experiment 2 — CPC  *(Mehari & Strodthoff 2022)*

**Pretrain**: full 10 s records, InfoNCE, AdamW wd 1e-3, constant LR. Model selection = lowest val (fold 9) InfoNCE loss. Checkpoint har 10 epochs → kernel restart pe resume.

**Two-step finetune** (unka key finding):
1. sirf head train (50 epochs), encoder/LSTM frozen par encoder BN statistics update hote rehte hain
2. full model 20 epochs, discriminative LR: head → LSTM ÷10 → encoder ÷100, reduced base LR

2.5 s windows, TTA mean.

**GPU memory**: 10 s × 512-d LSTM heavy hai. `fit_batch_size` khud check karta hai kitna micro-batch GPU mein fit hota hai, aur gradient accumulation se effective batch `pretrain_bs` hi rehta hai — recipe paper wali, sirf memory kam. Loss bhi har sequence ke `n_anchor` random timesteps pe compute hota hai (unbiased estimate).

In [12]:
def pretrain_ssl(name, model, loss_fn, opt, sched_fn, epochs, bs, save_every=10, select='val', accum=1):
    # bs = micro-batch actually on the GPU; effective batch = bs * accum (gradient accumulation)
    ckpt = OUT_DIR / f'{name}_pretrain_ckpt.pt'; final = OUT_DIR / f'{name}_pretrained.pt'
    if final.exists() and not FORCE_RETRAIN:
        print(f'[{name}] using pretrained weights {final}'); model.load_state_dict(torch.load(final, map_location=DEVICE)); return model
    it = WindowBatches(X_unl, bs=bs); itv = WindowBatches(X_unl_val, bs=bs, shuffle=False, drop_last=False)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, sched_fn(epochs * math.ceil(len(it) / accum))) if sched_fn else None
    scaler = make_scaler(); start, best, best_state = 0, float('inf'), None
    if ckpt.exists() and not FORCE_RETRAIN:
        s = torch.load(ckpt, map_location=DEVICE)
        model.load_state_dict(s['model']); opt.load_state_dict(s['opt']); start, best = s['epoch'], s['best']
        best_state = s.get('best_state');  sch is not None and sch.load_state_dict(s['sched'])
        print(f'[{name}] resuming pretraining from epoch {start}')
    for ep in range(start, epochs):
        model.train(); t0, run = time.time(), 0.
        opt.zero_grad(set_to_none=True)
        for i, xb in enumerate(it):
            with autocast(): loss = loss_fn(model, xb)
            scaler.scale(loss / accum).backward()
            if (i + 1) % accum == 0 or i + 1 == len(it):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if sch is not None: sch.step()
            run += loss.item()
        model.eval(); vl = 0.
        with torch.no_grad():
            for xb in itv:
                with autocast(): vl += loss_fn(model, xb).item() * len(xb)
        vl /= len(X_unl_val)
        flag = ''
        if select == 'val' and vl < best: best, best_state, flag = vl, copy.deepcopy(model.state_dict()), ' *'
        print(f'[{name}-pretrain] ep {ep+1:4d}/{epochs} train {run/len(it):.4f} val {vl:.4f}{flag} ({time.time()-t0:.0f}s)')
        if (ep + 1) % save_every == 0 or ep + 1 == epochs:
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep + 1, best=best, best_state=best_state,
                            sched=sch.state_dict() if sch is not None else None), ckpt)
    if select == 'val' and best_state is not None: model.load_state_dict(best_state)
    torch.save(model.state_dict(), final)
    return model

def fit_batch_size(model, loss_fn, bs, L, min_bs=4):
    # Try one forward/backward at `bs`; halve on CUDA OOM. Model weights/BN stats are restored afterwards.
    if DEVICE.type != 'cuda': return bs
    state = copy.deepcopy(model.state_dict()); model.train()
    while bs >= min_bs:
        x = loss = None
        try:
            x = torch.randn(bs, 12, L, device=DEVICE)
            with autocast(): loss = loss_fn(model, x)
            loss.backward()
            del x, loss; model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            model.load_state_dict(state)
            print(f'batch size {bs} fits in GPU memory'); return bs
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if 'out of memory' not in str(e).lower() and not isinstance(e, torch.cuda.OutOfMemoryError): raise
            del x, loss; model.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            print(f'batch size {bs} -> OOM, trying {bs // 2}'); bs //= 2
    raise RuntimeError('Even the minimum batch size does not fit; lower CFG_CPC["n_anchor"] or use a smaller d')

def bce_probe(m, x):
    return F.binary_cross_entropy_with_logits(m(x).float(), torch.zeros(len(x), N_CLASSES, device=x.device))

def exp_cpc():
    c = CFG_CPC; win = int(c['window_s'] * SAMPLING_RATE)
    model = CPCModel(d=c['d'], steps=c['steps']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=c['pretrain_lr'], weight_decay=c['wd'])
    cpc_fn = lambda m, x: m.cpc_loss(x, c['n_neg'], c['n_anchor'])[0]
    micro = fit_batch_size(model, cpc_fn, c['pretrain_bs'], SIG_LEN)
    accum = math.ceil(c['pretrain_bs'] / micro)
    print(f'CPC pretraining: micro-batch {micro} x {accum} accumulation steps = effective batch {micro * accum}')
    model = pretrain_ssl('cpc', model, cpc_fn, opt, None, c['pretrain_epochs'], micro, select='val', accum=accum)

    model.add_head(N_CLASSES); model.to(DEVICE)
    ft_bs = fit_batch_size(model, bce_probe, c['ft_bs'], win)
    # step 1: head only; BN running stats in the encoder keep updating because model.train() is used
    for p in list(model.encoder.parameters()) + list(model.rnn.parameters()) + list(model.proj.parameters()):
        p.requires_grad_(False)
    groups = [dict(params=model.head.parameters(), lr=c['ft_lr'], weight_decay=c['wd'])]
    model, _ = train_classifier(model, name='cpc_ft_head', param_groups=groups, epochs=c['head_epochs'], bs=ft_bs,
                                win=win, sched='constant', agg='mean')
    # step 2: everything, discriminative LRs (factor 10 per group), reduced LR
    for p in list(model.encoder.parameters()) + list(model.rnn.parameters()):
        p.requires_grad_(True)
    lr2 = c['ft_lr'] / 10
    groups = [dict(params=model.head.parameters(), lr=lr2, weight_decay=c['wd']),
              dict(params=model.rnn.parameters(), lr=lr2 / 10, weight_decay=c['wd']),
              dict(params=model.encoder.parameters(), lr=lr2 / 100, weight_decay=c['wd'])]
    model, _ = train_classifier(model, name='cpc_ft_full', param_groups=groups, epochs=c['full_epochs'], bs=ft_bs,
                                win=win, sched='constant', agg='mean')
    torch.save(model.state_dict(), OUT_DIR / 'cpc_finetuned.pt')
    return predict(model, Xva, win, agg='mean'), predict(model, Xte, win, agg='mean')

if RUN['cpc']:
    run_experiment('cpc', exp_cpc)

batch size 64 fits in GPU memory
CPC pretraining: micro-batch 64 x 1 accumulation steps = effective batch 64


RuntimeError: cuDNN error: CUDNN_STATUS_EXECUTION_FAILED_CUDART

## Experiment 3 — MTECG masked transformer  *(Zhou et al.)*

**Pretrain**: AdamW (β=0.9, 0.95), wd 0.05, LR 1e-3, batch 256, cosine + 40 warmup epochs, 25% mask, per-segment-norm target. Final epoch ka encoder use hota hai (MAE convention).

**Finetune**: AdamW (β=0.9, 0.999), wd 0.05 (bias/norm/pos par 0), LR 1e-3, layer-wise decay 0.6, DropPath 0.4, cosine + 5 warmup, 50 epochs, BCE. Input = full 10 s (single window).

In [ ]:
def mtecg_param_groups(model, lr, wd, layer_decay):
    n_layers = len(model.blocks) + 1
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if n in ('cls_token', 'pos_embed') or n.startswith('patch_embed'): lid = 0
        elif n.startswith('blocks.'): lid = int(n.split('.')[1]) + 1
        else: lid = n_layers
        no_wd = p.ndim == 1 or n in ('cls_token', 'pos_embed')
        key = (lid, no_wd)
        if key not in groups:
            groups[key] = dict(params=[], lr=lr * layer_decay ** (n_layers - lid), weight_decay=0. if no_wd else wd)
        groups[key]['params'].append(p)
    return list(groups.values())

def exp_mtecg():
    c = CFG_MTECG
    arch = dict(seq_len=SIG_LEN, patch=MT_PATCH, dim=c['dim'], depth=c['depth'], heads=c['heads'])
    pre = MTECG(**arch, with_decoder=True, dec_dim=c['dec_dim'], dec_depth=c['dec_depth'], dec_heads=c['dec_heads']).to(DEVICE)
    decay, no_decay = [], []
    for n, p in pre.named_parameters():
        (no_decay if p.ndim == 1 or n in ('cls_token', 'pos_embed', 'mask_token', 'dec_pos') else decay).append(p)
    opt = torch.optim.AdamW([dict(params=decay, weight_decay=c['pretrain_wd']), dict(params=no_decay, weight_decay=0.)],
                            lr=c['pretrain_lr'], betas=(0.9, 0.95))
    mae_fn = lambda m, x: m.mae_loss(x, c['mask_ratio'], c['target'])
    micro = fit_batch_size(pre, mae_fn, c['pretrain_bs'], SIG_LEN)
    accum = math.ceil(c['pretrain_bs'] / micro)
    print(f'MTECG pretraining: micro-batch {micro} x {accum} accumulation steps = effective batch {micro * accum}')
    n_steps = math.ceil(len(WindowBatches(X_unl, bs=micro)) / accum)          # optimizer steps per epoch
    pre = pretrain_ssl('mtecg', pre, mae_fn, opt, lambda total: cosine_warmup(total, c['pretrain_warmup'] * n_steps),
                       c['pretrain_epochs'], micro, select='last', accum=accum)

    model = MTECG(**arch, drop_path=c['drop_path'], n_out=N_CLASSES, with_decoder=False)
    enc_state = {k: v for k, v in pre.state_dict().items() if not k.startswith(('dec_', 'mask_token'))}
    missing, unexpected = model.load_state_dict(enc_state, strict=False)
    print('loaded pretrained encoder | newly initialised:', missing)
    del pre
    model.to(DEVICE); ft_bs = fit_batch_size(model, bce_probe, c['ft_bs'], SIG_LEN)
    groups = mtecg_param_groups(model, c['ft_lr'] * ft_bs / c['ft_bs'], c['ft_wd'], c['layer_decay'])   # linear LR scaling if bs shrank
    model, _ = train_classifier(model, name='mtecg_ft', param_groups=groups, epochs=c['ft_epochs'], bs=ft_bs,
                                win=SIG_LEN, sched='cosine', warmup_epochs=c['ft_warmup'], betas=(0.9, 0.999), agg='max')
    torch.save(model.state_dict(), OUT_DIR / 'mtecg_finetuned.pt')
    return predict(model, Xva, SIG_LEN), predict(model, Xte, SIG_LEN)

if RUN['mtecg']:
    run_experiment('mtecg', exp_mtecg)

## Final evaluation — ensemble, bootstrap CIs, paired significance  *(Strodthoff 2021)*

- **Ensemble** = trained models ki probabilities ka mean (benchmark ka approach). Do ensembles report hote hain: sab models, aur sirf SSL models.
- Har model ke liye: test macro AUC + 95% CI, aur best single model ke against paired bootstrap difference. Agar Δ ka CI 0 cross karta hai → difference **significant nahi** (paper mein yahi likhna).

In [ ]:
names = [n for n in RESULTS if not n.startswith('ensemble')]
if len(names) >= 2:
    RESULTS['ensemble_all'] = dict(val=np.mean([RESULTS[n]['val'] for n in names], 0),
                                   test=np.mean([RESULTS[n]['test'] for n in names], 0))
ssl = [n for n in ('cpc', 'mtecg') if n in RESULTS]
if len(ssl) == 2:
    RESULTS['ensemble_ssl'] = dict(val=np.mean([RESULTS[n]['val'] for n in ssl], 0),
                                   test=np.mean([RESULTS[n]['test'] for n in ssl], 0))

print(f'bootstrapping test set ({N_BOOTSTRAP} paired resamples)...')
boot_idx = bootstrap_indices(Y_test_np, N_BOOTSTRAP)
boot = {n: np.array([macro_auc(Y_test_np[i], r['test'][i]) for i in boot_idx]) for n, r in tqdm(RESULTS.items())}

rows = []
for n, r in RESULTS.items():
    th = tune_thresholds(Y_val_np, r['val'])
    tm = threshold_metrics(Y_test_np, r['test'] >= th[None])
    lo, hi = np.percentile(boot[n], [2.5, 97.5])
    rows.append(dict(model=n, val_macroAUC=macro_auc(Y_val_np, r['val']), test_macroAUC=macro_auc(Y_test_np, r['test']),
                     ci95_low=lo, ci95_high=hi, **{f'test_{k}': v for k, v in tm.items()}))
res = pd.DataFrame(rows).sort_values('test_macroAUC', ascending=False).reset_index(drop=True)

best_single = res[~res.model.str.startswith('ensemble')].iloc[0].model
d_rows = []
for n in RESULTS:
    if n == best_single: continue
    d = boot[n] - boot[best_single]; lo, hi = np.percentile(d, [2.5, 97.5])
    d_rows.append(dict(model=n, vs=best_single, delta_AUC=d.mean(), ci95_low=lo, ci95_high=hi, significant=not (lo <= 0 <= hi)))
diff = pd.DataFrame(d_rows)

res.to_csv(OUT_DIR / 'results.csv', index=False); diff.to_csv(OUT_DIR / 'paired_differences.csv', index=False)
pd.set_option('display.float_format', '{:.4f}'.format)
print(f'TASK={TASK} | {N_CLASSES} classes | test fold n={len(Y_test_np)}')
display(res)
print(f'Paired bootstrap vs best single model ({best_single}):')
display(diff)

In [ ]:
# Per-label test AUC for every model (sorted by best model) -> where the gains come from / which labels stay hard
best_name = res.iloc[0].model
pcl = pd.DataFrame({n: per_class_auc(Y_test_np, r['test']) for n, r in RESULTS.items()}, index=CLASSES)
pcl.insert(0, 'n_test_pos', Y_test_np.sum(0).astype(int))
pcl = pcl.sort_values(best_name)
pcl.to_csv(OUT_DIR / 'per_label_auc.csv')
print('Hardest 15 labels for', best_name); display(pcl.head(15))

hist_files = sorted(OUT_DIR.glob('*_history.csv'))
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if hist_files and plt is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    for f in hist_files:
        h = pd.read_csv(f); ax.plot(h.epoch, h.val_auc, label=f.stem.replace('_history', ''))
    ax.set_xlabel('epoch'); ax.set_ylabel('val macro AUC'); ax.grid(alpha=.3); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / 'val_curves.png', dpi=150); plt.show()
print('All outputs in', OUT_DIR.resolve())